In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Comparison Analysis\n",
    "## Python Screening vs Covidence Gold Standard\n",
    "\n",
    "This notebook compares Python screening results with Covidence decisions."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import libraries\n",
    "import sys\n",
    "sys.path.append('../src')\n",
    "\n",
    "from comparison import ScreeningComparator\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import json\n",
    "import yaml\n",
    "from pathlib import Path\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Visualization setup\n",
    "plt.style.use('seaborn-v0_8-whitegrid')\n",
    "sns.set_palette(\"Set2\")\n",
    "\n",
    "# Display options\n",
    "pd.set_option('display.max_columns', None)\n",
    "pd.set_option('display.max_rows', 100)\n",
    "pd.set_option('display.max_colwidth', 150)\n",
    "\n",
    "print(\"✅ Libraries imported\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Configuration\n",
    "REVIEWS = {\n",
    "    'insulin': {\n",
    "        'python_csv': '../data/output/insulin_screening/screening_decisions_insulin.csv',\n",
    "        'covidence_csv': '../data/insulin/covidence_included.csv',\n",
    "        'config': '../config/insulin_config.yaml'\n",
    "    },\n",
    "    'glp1': {\n",
    "        'python_csv': '../data/output/glp1_screening/screening_decisions_glp1.csv',\n",
    "        'covidence_csv': '../data/glp1/covidence_included.csv',\n",
    "        'config': '../config/glp1_config.yaml'\n",
    "    }\n",
    "}\n",
    "\n",
    "# Select review\n",
    "selected_review = 'insulin'  # Change to 'glp1' for GLP-1 analysis\n",
    "review_config = REVIEWS[selected_review]\n",
    "\n",
    "print(f\"🔍 Selected Review: {selected_review.upper()}\")\n",
    "print(f\"Python CSV: {review_config['python_csv']}\")\n",
    "print(f\"Covidence CSV: {review_config['covidence_csv']}\")\n",
    "\n",
    "# Load configuration\n",
    "with open(review_config['config'], 'r') as f:\n",
    "    config = yaml.safe_load(f)\n",
    "print(f\"Review: {config['review_name']}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load Python screening results\n",
    "python_df = pd.read_csv(review_config['python_csv'])\n",
    "print(f\"Python data: {python_df.shape} rows × {python_df.shape[1]} columns\")\n",
    "\n",
    "# Load Covidence data\n",
    "try:\n",
    "    covidence_df = pd.read_csv(review_config['covidence_csv'], encoding='utf-8')\n",
    "except UnicodeDecodeError:\n",
    "    covidence_df = pd.read_csv(review_config['covidence_csv'], encoding='latin-1')\n",
    "\n",
    "print(f\"Covidence data: {covidence_df.shape} rows × {covidence_df.shape[1]} columns\")\n",
    "\n",
    "# Standardize column names\n",
    "python_df.columns = [col.strip().lower().replace(' ', '_') for col in python_df.columns]\n",
    "covidence_df.columns = [col.strip().lower().replace(' ', '_') for col in covidence_df.columns]\n",
    "\n",
    "print(f\"\\nPython columns: {list(python_df.columns)[:10]}...\")\n",
    "print(f\"Covidence columns: {list(covidence_df.columns)[:10]}...\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Initialize Comparator"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize comparator with different matching strategies\n",
    "comparators = {\n",
    "    'accession_doi': ScreeningComparator(matching_strategy='accession_doi'),\n",
    "    'doi_only': ScreeningComparator(matching_strategy='doi_only'),\n",
    "    'accession_only': ScreeningComparator(matching_strategy='accession_only')\n",
    "}\n",
    "\n",
    "print(\"🔧 Available matching strategies:\")\n",
    "for strategy, comparator in comparators.items():\n",
    "    print(f\"  • {strategy}\")\n",
    "\n",
    "# Select strategy\n",
    "selected_strategy = 'accession_doi'\n",
    "comparator = comparators[selected_strategy]\n",
    "print(f\"\\nSelected matching strategy: {selected_strategy}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Run Comparison"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Run comparison\n",
    "print(\"🔍 Running comparison...\")\n",
    "\n",
    "# Filter Python to include+maybe\n",
    "if 'decision' in python_df.columns:\n",
    "    python_include_maybe = python_df[python_df['decision'].isin(['include', 'maybe'])]\n",
    "    print(f\"Python include+maybe: {len(python_include_maybe)} records\")\n",
    "else:\n",
    "    python_include_maybe = python_df\n",
    "    print(\"Warning: No 'decision' column in Python data\")\n",
    "\n",
    "# Filter Covidence to included\n",
    "covidence_included = covidence_df.copy()\n",
    "if 'decision' in covidence_df.columns:\n",
    "    covidence_included = covidence_df[covidence_df['decision'].str.contains('include', case=False, na=False)]\n",
    "    print(f\"Covidence included: {len(covidence_included)} records\")\n",
    "else:\n",
    "    print(\"Warning: No 'decision' column in Covidence data, using all records\")\n",
    "\n",
    "# Run comparison\n",
    "comparison_results = comparator.compare(\n",
    "    review_config['python_csv'],\n",
    "    review_config['covidence_csv']\n",
    ")\n",
    "\n",
    "print(f\"\\n📊 COMPARISON RESULTS ({selected_strategy} matching):\")\n",
    "print(\"=\" * 50)\n",
    "print(f\"Common studies: {comparison_results['common_studies']:,}\")\n",
    "print(f\"Python only: {comparison_results['python_only']:,}\")\n",
    "print(f\"Covidence only: {comparison_results['covidence_only']:,}\")\n",
    "print(f\"Python total (include+maybe): {comparison_results['python_total']:,}\")\n",
    "print(f\"Covidence total (included): {comparison_results['covidence_total']:,}\")\n",
    "\n",
    "if comparison_results['covidence_total'] > 0:\n",
    "    agreement = comparison_results['common_studies'] / comparison_results['covidence_total']\n",
    "    print(f\"Agreement rate: {agreement:.1%}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Visualize Comparison"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create visualizations\n",
    "fig, axes = plt.subplots(2, 2, figsize=(14, 10))\n",
    "\n",
    "# 1. Venn diagram-like visualization\n",
    "categories = ['Python & Covidence', 'Python Only', 'Covidence Only']\n",
    "values = [\n",
    "    comparison_results['common_studies'],\n",
    "    comparison_results['python_only'],\n",
    "    comparison_results['covidence_only']\n",
    "]\n",
    "colors = ['#3498db', '#2ecc71', '#e74c3c']\n",
    "\n",
    "bars = axes[0, 0].bar(categories, values, color=colors, edgecolor='black')\n",
    "axes[0, 0].set_title('Study Distribution', fontsize=14, fontweight='bold')\n",
    "axes[0, 0].set_ylabel('Number of Studies')\n",
    "axes[0, 0].tick_params(axis='x', rotation=45)\n",
    "\n",
    "# Add value labels\n",
    "for bar, value in zip(bars, values):\n",
    "    height = bar.get_height()\n",
    "    axes[0, 0].text(bar.get_x() + bar.get_width()/2., height + 5,\n",
    "                   f'{value:,}', ha='center', va='bottom', fontweight='bold')\n",
    "\n",
    "# 2. Agreement metrics\n",
    "agreement_metrics = {\n",
    "    'Sensitivity': comparison_results['common_studies'] / comparison_results['covidence_total'] if comparison_results['covidence_total'] > 0 else 0,\n",
    "    'Coverage': comparison_results['common_studies'] / comparison_results['python_total'] if comparison_results['python_total'] > 0 else 0\n",
    "}\n",
    "\n",
    "metric_names = list(agreement_metrics.keys())\n",
    "metric_values = list(agreement_metrics.values())\n",
    "\n",
    "bars = axes[0, 1].bar(metric_names, metric_values, color=['#9b59b6', '#1abc9c'])\n",
    "axes[0, 1].set_ylim(0, 1)\n",
    "axes[0, 1].set_title('Agreement Metrics', fontsize=14, fontweight='bold')\n",
    "axes[0, 1].set_ylabel('Score')\n",
    "axes[0, 1].axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='Target (80%)')\n",
    "axes[0, 1].legend()\n",
    "\n",
    "# Add percentage labels\n",
    "for bar, value in zip(bars, metric_values):\n",
    "    height = bar.get_height()\n",
    "    axes[0, 1].text(bar.get_x() + bar.get_width()/2., height + 0.02,\n",
    "                   f'{value:.1%}', ha='center', va='bottom', fontweight='bold')\n",
    "\n",
    "# 3. Matching strategy comparison\n",
    "print(\"\\n🔄 Comparing matching strategies...\")\n",
    "\n",
    "strategy_results = {}\n",
    "for strategy_name, strategy_comparator in comparators.items():\n",
    "    try:\n",
    "        results = strategy_comparator.compare(\n",
    "            review_config['python_csv'],\n",
    "            review_config['covidence_csv']\n",
    "        )\n",
    "        strategy_results[strategy_name] = results['common_studies']\n",
    "    except Exception as e:\n",
    "        print(f\"  Error with {strategy_name}: {e}\")\n",
    "        strategy_results[strategy_name] = 0\n",
    "\n",
    "if strategy_results:\n",
    "    strategy_names = list(strategy_results.keys())\n",
    "    strategy_values = list(strategy_results.values())\n",
    "    \n",
    "    bars = axes[1, 0].bar(strategy_names, strategy_values, color='#f39c12')\n",
    "    axes[1, 0].set_title('Matching Strategy Performance', fontsize=14, fontweight='bold')\n",
    "    axes[1, 0].set_ylabel('Common Studies Found')\n",
    "    axes[1, 0].tick_params(axis='x', rotation=45)\n",
    "    \n",
    "    # Add value labels\n",
    "    for bar, value in zip(bars, strategy_values):\n",
    "        height = bar.get_height()\n",
    "        axes[1, 0].text(bar.get_x() + bar.get_width()/2., height + 1,\n",
    "                       f'{value:,}', ha='center', va='bottom', fontweight='bold')\n",
    "\n",
    "    # Highlight best strategy\n",
    "    best_strategy = max(strategy_results, key=strategy_results.get)\n",
    "    best_idx = strategy_names.index(best_strategy)\n",
    "    bars[best_idx].set_color('#e74c3c')\n",
    "    axes[1, 0].text(bars[best_idx].get_x() + bars[best_idx].get_width()/2., \n",
    "                   bars[best_idx].get_height() + max(strategy_values)*0.05,\n",
    "                   'BEST', ha='center', va='bottom', fontweight='bold', color='#e74c3c')\n",
    "\n",
    "# 4. Summary statistics\n",
    "axes[1, 1].axis('off')\n",
    "axes[1, 1].set_title('Comparison Summary', fontsize=14, fontweight='bold', pad=20)\n",
    "\n",
    "summary_text = f\"\"\"\n",
    "Review: {config['review_name']}\n",
    "\n",
    "Matching Strategy: {selected_strategy}\n",
    "\n",
    "Python Screening:\n",
    "• Include+Maybe: {comparison_results['python_total']:,}\n",
    "• Exclude: {comparison_results['python_total'] - len(python_include_maybe):,}\n",
    "\n",
    "Covidence Gold Standard:\n",
    "• Included: {comparison_results['covidence_total']:,}\n",
    "\n",
    "Comparison Results:\n",
    "• Common Studies: {comparison_results['common_studies']:,}\n",
    "• Python Only: {comparison_results['python_only']:,}\n",
    "• Covidence Only: {comparison_results['covidence_only']:,}\n",
    "\n",
    "Agreement: {agreement:.1%}\n",
    "\n",
    "Best Matching: {best_strategy if 'best_strategy' in locals() else 'N/A'}\n",
    "\"\"\"\n",
    "\n",
    "axes[1, 1].text(0.05, 0.95, summary_text, transform=axes[1, 1].transAxes,\n",
    "               fontsize=10, family='monospace', verticalalignment='top')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Detailed Analysis of Discrepancies"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create match keys for detailed analysis\n",
    "print(\"🔎 Analyzing discrepancies...\")\n",
    "\n",
    "# Prepare dataframes with match keys\n",
    "python_df['match_key'] = comparator._create_match_keys(python_df)\n",
    "covidence_df['match_key'] = comparator._create_match_keys(covidence_df)\n",
    "\n",
    "# Find matches\n",
    "python_keys = set(python_df['match_key'])\n",
    "covidence_keys = set(covidence_df['match_key'])\n",
    "common_keys = python_keys.intersection(covidence_keys)\n",
    "\n",
    "# Filter out 'no_id' keys\n",
    "real_common_keys = [k for k in common_keys if not k.startswith('no_id:')]\n",
    "print(f\"Real matches (with identifiers): {len(real_common_keys)}\")\n",
    "\n",
    "# Get matched records\n",
    "common_records = python_df[python_df['match_key'].isin(real_common_keys)].copy()\n",
    "print(f\"Common records found: {len(common_records)}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Analyze decisions for common records\n",
    "if 'decision' in common_records.columns and len(common_records) > 0:\n",
    "    print(\"\\n📋 Decisions for Common Records:\")\n",
    "    decision_counts = common_records['decision'].value_counts()\n",
    "    \n",
    "    for decision, count in decision_counts.items():\n",
    "        percentage = count / len(common_records) * 100\n",
    "        print(f\"  {decision}: {count} records ({percentage:.1f}%)\")\n",
    "    \n",
    "    # Show sample of 'maybe' decisions that matched\n",
    "    if 'maybe' in decision_counts:\n",
    "        maybe_records = common_records[common_records['decision'] == 'maybe']\n",
    "        print(f\"\\n🔍 Sample 'maybe' decisions that matched Covidence (first 3):\")\n",
    "        for i, (idx, row) in enumerate(maybe_records.head(3).iterrows(), 1):\n",
    "            print(f\"  {i}. Title: {row.get('title', 'No title')[:80]}...\")\n",
    "            if 'reason' in row:\n",
    "                print(f\"     Reason: {row['reason']}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Find Python-only and Covidence-only studies\n",
    "python_only_keys = python_keys - covidence_keys\n",
    "covidence_only_keys = covidence_keys - python_keys\n",
    "\n",
    "# Filter out 'no_id' keys\n",
    "real_python_only = [k for k in python_only_keys if not k.startswith('no_id:')]\n",
    "real_covidence_only = [k for k in covidence_only_keys if not k.startswith('no_id:')]\n",
    "\n",
    "print(f\"\\n📊 Discrepancy Analysis:\")\n",
    "print(f\"Python-only studies (with identifiers): {len(real_python_only)}\")\n",
    "print(f\"Covidence-only studies (with identifiers): {len(real_covidence_only)}\")\n",
    "\n",
    "# Analyze Python-only studies\n",
    "if len(real_python_only) > 0 and 'decision' in python_df.columns:\n",
    "    python_only_df = python_df[python_df['match_key'].isin(real_python_only)]\n",
    "    \n",
    "    print(f\"\\n🔍 Python-only studies analysis:\")\n",
    "    decision_counts = python_only_df['decision'].value_counts()\n",
    "    \n",
    "    for decision, count in decision_counts.items():\n",
    "        percentage = count / len(python_only_df) * 100\n",
    "        print(f\"  {decision}: {count} records ({percentage:.1f}%)\")\n",
    "    \n",
    "    # Show reasons for include/maybe decisions\n",
    "    if 'include' in decision_counts or 'maybe' in decision_counts:\n",
    "        include_maybe = python_only_df[python_only_df['decision'].isin(['include', 'maybe'])]\n",
    "        if 'reason' in include_maybe.columns:\n",
    "            print(f\"\\n📝 Common reasons for Python include/maybe (Covidence excluded):\")\n",
    "            reason_counts = include_maybe['reason'].value_counts().head(5)\n",
    "            for reason, count in reason_counts.items():\n",
    "                print(f\"  • {reason}: {count}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Save Comparison Results"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save detailed comparison results\n",
    "output_dir = Path(f'../data/output/{selected_review}_comparison')\n",
    "output_dir.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "print(f\"\\n💾 Saving comparison results to: {output_dir}\")\n",
    "\n",
    "# Save comparison results\n",
    "results_file = output_dir / 'comparison_results.json'\n",
    "with open(results_file, 'w') as f:\n",
    "    json.dump(comparison_results, f, indent=2)\n",
    "print(f\"  • {results_file.name}\")\n",
    "\n",
    "# Save common studies\n",
    "if len(common_records) > 0:\n",
    "    common_studies_file = output_dir / 'common_studies.csv'\n",
    "    common_records.to_csv(common_studies_file, index=False)\n",
    "    print(f\"  • {common_studies_file.name} ({len(common_records)} records)\")\n",
    "\n",
    "# Save Python-only studies\n",
    "if 'real_python_only' in locals() and len(real_python_only) > 0:\n",
    "    python_only_studies = python_df[python_df['match_key'].isin(real_python_only)]\n",
    "    python_only_file = output_dir / 'python_only_studies.csv'\n",
    "    python_only_studies.to_csv(python_only_file, index=False)\n",
    "    print(f\"  • {python_only_file.name} ({len(python_only_studies)} records)\")\n",
    "\n",
    "# Save Covidence-only studies\n",
    "if 'real_covidence_only' in locals() and len(real_covidence_only) > 0:\n",
    "    covidence_only_studies = covidence_df[covidence_df['match_key'].isin(real_covidence_only)]\n",
    "    covidence_only_file = output_dir / 'covidence_only_studies.csv'\n",
    "    covidence_only_studies.to_csv(covidence_only_file, index=False)\n",
    "    print(f\"  • {covidence_only_file.name} ({len(covidence_only_studies)} records)\")\n",
    "\n",
    "# Save comparison report\n",
    "report = {\n",
    "    'review_name': config['review_name'],\n",
    "    'analysis_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),\n",
    "    'matching_strategy': selected_strategy,\n",
    "    'comparison_results': comparison_results,\n",
    "    'summary': {\n",
    "        'total_common_studies': len(real_common_keys),\n",
    "        'agreement_rate': agreement if 'agreement' in locals() else 0,\n",
    "        'python_only_with_ids': len(real_python_only),\n",
    "        'covidence_only_with_ids': len(real_covidence_only),\n",
    "        'identifier_stats': {\n",
    "            'python_with_doi': python_df['match_key'].str.startswith('doi:').sum(),\n",
    "            'python_with_accession': python_df['match_key'].str.startswith('accession:').sum(),\n",
    "            'python_no_id': python_df['match_key'].str.startswith('no_id:').sum(),\n",
    "            'covidence_with_doi': covidence_df['match_key'].str.startswith('doi:').sum(),\n",
    "            'covidence_with_accession': covidence_df['match_key'].str.startswith('accession:').sum(),\n",
    "            'covidence_no_id': covidence_df['match_key'].str.startswith('no_id:').sum(),\n",
    "        }\n",
    "    }\n",
    "}\n",
    "\n",
    "report_file = output_dir / 'comparison_report.json'\n",
    "with open(report_file, 'w') as f:\n",
    "    json.dump(report, f, indent=2)\n",
    "print(f\"  • {report_file.name}\")\n",
    "\n",
    "print(f\"\\n✅ Comparison analysis complete!\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "cochrane-screening",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}